# Mall World Swedenborgian Hypothesis Testing

**Purpose:** Rigorous statistical testing of five hypotheses derived from Swedenborgian cosmology against phenomenological data extracted from r/TheMallWorld dream reports.

**Dataset:** ~4,500+ extracted Mall World dream posts (filtered to actual dream reports)

**Date:** January 20, 2026

---

## Research Framework

This notebook tests five specific predictions derived from Swedenborg's descriptions of the spiritual world:

1. **Hypothesis 1: Affectional Proximity** — Travel depends on emotional state, not physical locomotion
2. **Hypothesis 2: Somatic Sphere Conflict** — Physical distress correlates with "incompatible" environments
3. **Hypothesis 3: Digestive Sorting Mechanism** — Processing/sorting locations (schools, hospitals) are prevalent
4. **Hypothesis 4: Proprium Exposure** — Bathrooms show exposure (not privacy) and filth patterns
5. **Hypothesis 5: Plastic Heavens** — Luxury locations exhibit reality instability

---

## Methodological Principles

### Data Integrity
- **Filter first**: Only actual dream reports are analyzed (excluding AI images, theories, questions)
- **Report sample sizes**: Every test includes n for transparency
- **Use appropriate tests**: Chi-square for independence, Fisher's exact for small samples
- **Report effect sizes**: Cramér's V, odds ratios where applicable
- **Bonferroni correction**: For multiple comparisons within hypothesis

### Honest Reporting
- All results reported regardless of direction
- Negative findings are as important as positive ones
- No p-hacking: hypotheses defined a priori from research proposal
- "Not mentioned" values are tracked separately (missing data ≠ negative data)

### Interpretation Guidelines
- p < 0.001: Strong evidence
- p < 0.01: Moderate evidence
- p < 0.05: Weak evidence (requires replication)
- p ≥ 0.05: No evidence (null hypothesis retained)

## Setup: Load Libraries and Data

In [1]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from scipy.stats import chi2_contingency, fisher_exact, mannwhitneyu, spearmanr
from collections import Counter
from typing import List, Dict, Any, Optional, Tuple
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

print("✓ Libraries loaded")

✓ Libraries loaded


In [2]:
def load_mallworld_data(structured_path: Path) -> List[Dict]:
    """
    Load all extracted Mall World JSON files.
    Returns raw records for filtering.
    """
    records = []
    for json_file in sorted(structured_path.glob('*.json')):
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                record = json.load(f)
                records.append(record)
        except Exception as e:
            print(f"Error loading {json_file.name}: {e}")
    return records

# Load data
structured_path = Path('../structured')
raw_records = load_mallworld_data(structured_path)

print(f"✓ Loaded {len(raw_records):,} total records")

✓ Loaded 3,732 total records


In [3]:
# Examine post type distribution BEFORE filtering
post_types = Counter()
for r in raw_records:
    ext = r.get('extraction', {}) or {}
    classification = ext.get('classification', {}) or {}
    post_type = classification.get('post_type', 'unknown')
    post_types[post_type] += 1

print("="*70)
print("POST TYPE DISTRIBUTION (All Records)")
print("="*70)
print("\nThis shows why filtering is CRITICAL:")
print("-"*50)

for ptype, count in sorted(post_types.items(), key=lambda x: -x[1]):
    pct = count / len(raw_records) * 100
    marker = "✓ INCLUDE" if ptype in ['dream_report', 'dream_report_with_map'] else "✗ EXCLUDE"
    print(f"  {ptype:30s}: {count:5,} ({pct:5.1f}%) {marker}")

# Calculate includable
includable = post_types.get('dream_report', 0) + post_types.get('dream_report_with_map', 0)
print(f"\n✓ USABLE DREAM REPORTS: {includable:,} ({includable/len(raw_records)*100:.1f}%)")

POST TYPE DISTRIBUTION (All Records)

This shows why filtering is CRITICAL:
--------------------------------------------------
  dream_report                  : 1,792 ( 48.0%) ✓ INCLUDE
  question                      :   888 ( 23.8%) ✗ EXCLUDE
  ai_visualization              :   330 (  8.8%) ✗ EXCLUDE
  dream_report_with_map         :   138 (  3.7%) ✓ INCLUDE
  meta_discussion               :   132 (  3.5%) ✗ EXCLUDE
  media_reference               :   110 (  2.9%) ✗ EXCLUDE
  theory                        :   110 (  2.9%) ✗ EXCLUDE
  map_only                      :    66 (  1.8%) ✗ EXCLUDE
  introduction                  :    41 (  1.1%) ✗ EXCLUDE
  survey_research               :    38 (  1.0%) ✗ EXCLUDE
  shared_dream_claim            :    34 (  0.9%) ✗ EXCLUDE
  other                         :    32 (  0.9%) ✗ EXCLUDE
  lucid_technique               :    15 (  0.4%) ✗ EXCLUDE
  off_topic                     :     5 (  0.1%) ✗ EXCLUDE
  spam_troll                    :     1 (  0.0%

In [4]:
# Filter to only actual dream reports
VALID_POST_TYPES = {'dream_report', 'dream_report_with_map'}

def is_valid_dream_report(record: Dict) -> bool:
    """Check if record is an actual dream report worth analyzing."""
    ext = record.get('extraction', {}) or {}
    classification = ext.get('classification', {}) or {}
    
    # Must be a dream report type
    post_type = classification.get('post_type', '')
    if post_type not in VALID_POST_TYPES:
        return False
    
    # Must have extractable content
    if classification.get('should_skip_extraction', False):
        return False
    
    # Must have at least one location
    locations = ext.get('locations', [])
    if not locations or len(locations) == 0:
        return False
    
    return True

# Apply filter
valid_records = [r for r in raw_records if is_valid_dream_report(r)]

print(f"\n✓ After filtering: {len(valid_records):,} valid dream reports")
print(f"  (Excluded {len(raw_records) - len(valid_records):,} non-dream posts)")


✓ After filtering: 1,926 valid dream reports
  (Excluded 1,806 non-dream posts)


In [5]:
def flatten_location(loc: Dict, dream_id: str) -> Dict:
    """
    Flatten a location record for DataFrame analysis.
    Each location becomes a row with dream_id for grouping.
    """
    qualities = loc.get('qualities', {}) or {}
    position = loc.get('position', {}) or {}
    
    return {
        'dream_id': dream_id,
        'location_id': loc.get('location_id'),
        'location_type': loc.get('location_type', 'unspecified'),
        'location_name': loc.get('location_name'),
        
        # Position
        'vertical_position': position.get('vertical', 'not_mentioned'),
        'horizontal_position': position.get('horizontal', 'not_mentioned'),
        
        # Core qualities
        'light': qualities.get('light', 'not_mentioned'),
        'light_temperature': qualities.get('light_temperature', 'not_mentioned'),
        'time_of_day': qualities.get('time_of_day', 'not_mentioned'),
        'state': qualities.get('state', 'not_mentioned'),
        'atmosphere': qualities.get('atmosphere', 'not_mentioned'),
        'reality_stability': qualities.get('reality_stability', 'not_mentioned'),
        'crowding': qualities.get('crowding', 'not_mentioned'),
        'crowd_behavior': qualities.get('crowd_behavior', 'not_mentioned'),
        'intellectual_focus': qualities.get('intellectual_focus', 'not_mentioned'),
        'cleanliness': qualities.get('cleanliness', 'not_mentioned'),
        'privacy_status': qualities.get('privacy_status', 'not_mentioned'),
        'water_presence': qualities.get('water_presence', 'not_mentioned'),
        
        # Dreamer response
        'affective_response': loc.get('affective_response', 'not_mentioned'),
        'somatic_response': loc.get('somatic_response', 'none'),
        
        # Context
        'is_familiar': loc.get('is_familiar', 'not_mentioned'),
        'is_accessible': loc.get('is_accessible', 'not_mentioned'),
        'dreamer_role': loc.get('dreamer_role', 'not_mentioned'),
        'visit_order': loc.get('visit_order'),
        
        # Raw description for qualitative check
        'raw_description': qualities.get('raw_description', ''),
    }


def flatten_connection(conn: Dict, dream_id: str) -> Dict:
    """
    Flatten a connection record for DataFrame analysis.
    """
    return {
        'dream_id': dream_id,
        'from_location_id': conn.get('from_location_id'),
        'to_location_id': conn.get('to_location_id'),
        'connection_type': conn.get('connection_type', 'unspecified'),
        'transit_mode': conn.get('transit_mode', 'not_mentioned'),
        'direction': conn.get('direction', 'unknown'),
        'difficulty': conn.get('difficulty', 'not_mentioned'),
        'mechanism_function': conn.get('mechanism_function', 'not_applicable'),
        'outcome': conn.get('outcome', 'not_mentioned'),
        'qualities': conn.get('qualities', ''),
    }


def flatten_interaction(inter: Dict, dream_id: str) -> Dict:
    """
    Flatten an interaction record for DataFrame analysis.
    """
    return {
        'dream_id': dream_id,
        'location_id': inter.get('location_id'),
        'interaction_type': inter.get('interaction_type', 'other'),
        'description': inter.get('description', ''),
        'outcome': inter.get('outcome', 'not_mentioned'),
        'failure_type': inter.get('failure_type', 'not_applicable'),
        'entities_count': len(inter.get('entities_involved', [])),
    }


def extract_entities_from_interactions(interactions: List[Dict], dream_id: str) -> List[Dict]:
    """
    Extract all entities from interactions.
    """
    entities = []
    for inter in interactions:
        for entity in inter.get('entities_involved', []):
            entities.append({
                'dream_id': dream_id,
                'location_id': inter.get('location_id'),
                'entity_type': entity.get('entity_type', 'other'),
                'demeanor': entity.get('demeanor', 'not_mentioned'),
                'role': entity.get('role', 'not_mentioned'),
                'authority_nature': entity.get('authority_nature', 'not_mentioned'),
            })
    return entities

In [6]:
# Build DataFrames
locations_data = []
connections_data = []
interactions_data = []
entities_data = []
dreams_meta = []

for record in valid_records:
    dream_id = record.get('source_url', record.get('title', 'unknown'))
    ext = record.get('extraction', {}) or {}
    meta = ext.get('meta', {}) or {}
    
    # Dream-level metadata
    dreams_meta.append({
        'dream_id': dream_id,
        'title': record.get('title', ''),
        'map_coherence': meta.get('map_coherence', 'not_enough_info'),
        'recurrence': meta.get('recurrence', 'not_mentioned'),
        'time_flow': meta.get('time_flow', 'not_mentioned'),
        'dreamer_identity': meta.get('dreamer_identity', 'not_mentioned'),
        'initial_emotion': meta.get('initial_emotion'),
        'final_emotion': meta.get('final_emotion'),
        'extraction_confidence': ext.get('extraction_confidence', 'unknown'),
        'has_spatial_data': ext.get('has_spatial_data', False),
        'location_count': len(ext.get('locations', [])),
        'connection_count': len(ext.get('connections', [])),
        'interaction_count': len(ext.get('interactions', [])),
    })
    
    # Locations
    for loc in ext.get('locations', []):
        locations_data.append(flatten_location(loc, dream_id))
    
    # Connections
    for conn in ext.get('connections', []):
        connections_data.append(flatten_connection(conn, dream_id))
    
    # Interactions
    interactions = ext.get('interactions', [])
    for inter in interactions:
        interactions_data.append(flatten_interaction(inter, dream_id))
    
    # Entities from interactions
    entities_data.extend(extract_entities_from_interactions(interactions, dream_id))

# Create DataFrames
df_dreams = pd.DataFrame(dreams_meta)
df_locations = pd.DataFrame(locations_data)
df_connections = pd.DataFrame(connections_data)
df_interactions = pd.DataFrame(interactions_data)
df_entities = pd.DataFrame(entities_data)

print("="*70)
print("DATA SUMMARY")
print("="*70)
print(f"  Dreams:       {len(df_dreams):,}")
print(f"  Locations:    {len(df_locations):,}")
print(f"  Connections:  {len(df_connections):,}")
print(f"  Interactions: {len(df_interactions):,}")
print(f"  Entities:     {len(df_entities):,}")

DATA SUMMARY
  Dreams:       1,926
  Locations:    8,707
  Connections:  4,600
  Interactions: 6,075
  Entities:     3,704


In [7]:
# Utility functions for statistical testing

def cramers_v(contingency_table):
    """Calculate Cramér's V for effect size."""
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    n = contingency_table.sum().sum()
    min_dim = min(contingency_table.shape) - 1
    if min_dim == 0:
        return 0.0
    return np.sqrt(chi2 / (n * min_dim))


def chi_square_test(contingency_table, test_name=""):
    """
    Perform chi-square test with full reporting.
    Returns dict with all statistics.
    """
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    n = contingency_table.sum().sum()
    v = cramers_v(contingency_table)
    
    # Effect size interpretation
    if v < 0.1:
        effect = "negligible"
    elif v < 0.3:
        effect = "small"
    elif v < 0.5:
        effect = "medium"
    else:
        effect = "large"
    
    return {
        'test_name': test_name,
        'chi2': chi2,
        'p': p,
        'dof': dof,
        'n': n,
        'cramers_v': v,
        'effect_size': effect,
        'significant_001': p < 0.001,
        'significant_01': p < 0.01,
        'significant_05': p < 0.05,
    }


def report_chi_square(result: Dict):
    """Print formatted chi-square result."""
    print(f"\n{result['test_name']}")
    print("-" * 50)
    print(f"  n = {result['n']:,}")
    print(f"  χ² = {result['chi2']:.2f}, df = {result['dof']}, p = {result['p']:.4e}")
    print(f"  Cramér's V = {result['cramers_v']:.3f} ({result['effect_size']} effect)")
    
    if result['significant_001']:
        print("  ⚠️ HIGHLY SIGNIFICANT (p < 0.001)")
    elif result['significant_01']:
        print("  ⚠️ SIGNIFICANT (p < 0.01)")
    elif result['significant_05']:
        print("  ⚠️ WEAKLY SIGNIFICANT (p < 0.05)")
    else:
        print("  ✓ NOT SIGNIFICANT (p ≥ 0.05)")


def filter_not_mentioned(series, column_name="value"):
    """
    Filter out 'not_mentioned' and report how many were excluded.
    """
    total = len(series)
    not_mentioned = (series == 'not_mentioned').sum()
    filtered = series[series != 'not_mentioned']
    
    if not_mentioned > 0:
        print(f"  Note: {not_mentioned:,} ({not_mentioned/total*100:.1f}%) '{column_name}' values were 'not_mentioned'")
    
    return filtered


print("✓ Statistical utilities loaded")

✓ Statistical utilities loaded


---

# PRELIMINARY: Data Quality Assessment

Before hypothesis testing, we must understand data completeness and quality.

In [8]:
print("="*70)
print("DATA QUALITY ASSESSMENT")
print("="*70)

# Extraction confidence distribution
print("\n1. EXTRACTION CONFIDENCE")
print("-" * 50)
confidence_dist = df_dreams['extraction_confidence'].value_counts()
for conf, count in confidence_dist.items():
    print(f"  {conf}: {count:,} ({count/len(df_dreams)*100:.1f}%)")

# Location type distribution
print("\n2. LOCATION TYPE DISTRIBUTION")
print("-" * 50)
loc_type_dist = df_locations['location_type'].value_counts().head(20)
for loc_type, count in loc_type_dist.items():
    print(f"  {loc_type:30s}: {count:,} ({count/len(df_locations)*100:.1f}%)")

# Key field completeness
print("\n3. KEY FIELD COMPLETENESS (Locations)")
print("-" * 50)
key_fields = ['light_temperature', 'atmosphere', 'reality_stability', 
              'cleanliness', 'privacy_status', 'somatic_response', 'affective_response']
for field in key_fields:
    not_mentioned = (df_locations[field] == 'not_mentioned').sum()
    none_val = (df_locations[field] == 'none').sum()
    populated = len(df_locations) - not_mentioned - none_val
    print(f"  {field:25s}: {populated:,} populated ({populated/len(df_locations)*100:.1f}%)")

DATA QUALITY ASSESSMENT

1. EXTRACTION CONFIDENCE
--------------------------------------------------
  medium: 1,286 (66.8%)
  high: 610 (31.7%)
  low: 21 (1.1%)
  minimal: 9 (0.5%)

2. LOCATION TYPE DISTRIBUTION
--------------------------------------------------
  other                         : 2,297 (26.4%)
  mall                          : 879 (10.1%)
  mall_store                    : 590 (6.8%)
  city_street                   : 383 (4.4%)
  house                         : 361 (4.1%)
  school                        : 264 (3.0%)
  hotel                         : 255 (2.9%)
  restaurant                    : 192 (2.2%)
  beach                         : 161 (1.8%)
  parking_lot                   : 141 (1.6%)
  neighborhood                  : 133 (1.5%)
  apartment                     : 118 (1.4%)
  airport                       : 116 (1.3%)
  amusement_park                : 116 (1.3%)
  waterpark                     : 114 (1.3%)
  mall_food_court               : 112 (1.3%)
  theater   

In [9]:
# Connection data quality
print("\n4. CONNECTION DATA")
print("-" * 50)
if len(df_connections) > 0:
    print(f"  Total connections: {len(df_connections):,}")
    print(f"\n  Connection types:")
    conn_types = df_connections['connection_type'].value_counts().head(10)
    for ct, count in conn_types.items():
        print(f"    {ct:20s}: {count:,}")
    
    print(f"\n  Transit modes:")
    transit_modes = df_connections['transit_mode'].value_counts()
    for tm, count in transit_modes.items():
        print(f"    {tm:20s}: {count:,}")
    
    print(f"\n  Traversal outcomes:")
    outcomes = df_connections['outcome'].value_counts()
    for outcome, count in outcomes.items():
        print(f"    {outcome:20s}: {count:,}")
else:
    print("  ⚠️ No connection data available")


4. CONNECTION DATA
--------------------------------------------------
  Total connections: 4,600

  Connection types:
    outside_path        : 1,037
    door                : 660
    hallway             : 572
    other               : 533
    stairs              : 382
    unspecified         : 351
    vehicle             : 323
    elevator            : 166
    escalator           : 123
    tunnel              : 117

  Transit modes:
    directed_active     : 2,537
    passive             : 518
    not_mentioned       : 517
    wandering           : 363
    fleeing             : 356
    instant             : 173
    drifting            : 69
    struggle            : 67

  Traversal outcomes:
    succeeded           : 2,971
    not_mentioned       : 1,383
    partial             : 99
    not_attempted       : 53
    failed              : 49
    interrupted         : 37
    still_in_progress   : 8


---

# HYPOTHESIS 1: The Law of Affectional Proximity (Non-Locality)

**Swedenborgian Grounding:** In the spiritual world, travel depends on emotional state, not physical locomotion. "Local distance is of so little consequence... as soon as the bond was relaxed, the spirit was borne away."

**Predictions:**
1. Successful traversal correlates with DIRECTED_ACTIVE (intent) or PASSIVE (influx)
2. FAILED/LOOPING outcomes correlate with WANDERING or ANXIETY
3. Mechanical connections (elevators) frequently MALFUNCTION or behave UNPREDICTABLY

In [10]:
print("="*70)
print("HYPOTHESIS 1: AFFECTIONAL PROXIMITY")
print("="*70)

if len(df_connections) < 50:
    print("\n⚠️ INSUFFICIENT CONNECTION DATA for robust testing")
    print(f"   Only {len(df_connections)} connections available")
else:
    # Test 1.1: Transit Mode vs Outcome
    print("\n1.1 TRANSIT MODE → TRAVERSAL OUTCOME")
    print("-" * 50)
    
    # Filter to connections with known transit mode and outcome
    conn_analysis = df_connections[
        (df_connections['transit_mode'] != 'not_mentioned') &
        (df_connections['outcome'] != 'not_mentioned')
    ].copy()
    
    print(f"  Connections with transit mode AND outcome: {len(conn_analysis):,}")
    
    if len(conn_analysis) >= 30:
        # Create contingency table
        contingency = pd.crosstab(conn_analysis['transit_mode'], conn_analysis['outcome'])
        print(f"\n  Contingency table:")
        print(contingency)
        
        # Chi-square test
        if contingency.shape[0] > 1 and contingency.shape[1] > 1:
            result = chi_square_test(contingency, "Transit Mode vs Outcome")
            report_chi_square(result)
            
            # Specific prediction: DIRECTED_ACTIVE should have higher success rate
            print("\n  Prediction check: DIRECTED_ACTIVE should correlate with SUCCESS")
            if 'directed_active' in contingency.index and 'succeeded' in contingency.columns:
                directed_success = contingency.loc['directed_active', 'succeeded'] if 'succeeded' in contingency.columns else 0
                directed_total = contingency.loc['directed_active'].sum()
                directed_rate = directed_success / directed_total * 100 if directed_total > 0 else 0
                
                other_success = contingency.drop('directed_active', errors='ignore')['succeeded'].sum() if 'succeeded' in contingency.columns else 0
                other_total = contingency.drop('directed_active', errors='ignore').sum().sum()
                other_rate = other_success / other_total * 100 if other_total > 0 else 0
                
                print(f"    DIRECTED_ACTIVE success rate: {directed_rate:.1f}%")
                print(f"    Other modes success rate: {other_rate:.1f}%")
    else:
        print("  ⚠️ Insufficient data for contingency analysis")

HYPOTHESIS 1: AFFECTIONAL PROXIMITY

1.1 TRANSIT MODE → TRAVERSAL OUTCOME
--------------------------------------------------
  Connections with transit mode AND outcome: 3,132

  Contingency table:
outcome          failed  interrupted  not_attempted  partial  \
transit_mode                                                   
directed_active      32           24             25       48   
drifting              0            1              0        2   
fleeing               4            1              0        9   
instant               0            2              0        2   
passive               5            7              5       22   
struggle              6            1              0        7   
wandering             1            0              1        9   

outcome          still_in_progress  succeeded  
transit_mode                                   
directed_active                  5       1778  
drifting                         0         57  
fleeing                          

In [12]:
# Test 1.2: Mechanical connections (elevators/escalators) malfunction rate
print("\n1.2 MECHANICAL CONNECTION DYSFUNCTION")
print("-" * 50)

# Filter to mechanical connections
mechanical_types = ['elevator', 'escalator']
mechanical_conn = df_connections[df_connections['connection_type'].isin(mechanical_types)].copy()

print(f"  Total mechanical connections (elevators/escalators): {len(mechanical_conn):,}")

if len(mechanical_conn) >= 20:
    # Mechanism function distribution
    mech_func = mechanical_conn['mechanism_function'].value_counts()
    print(f"\n  Mechanism function distribution:")
    for func, count in mech_func.items():
        print(f"    {func:25s}: {count:,} ({count/len(mechanical_conn)*100:.1f}%)")
    
    # Calculate dysfunction rate
    dysfunctional = ['malfunctioning', 'broken', 'unpredictable', 'dangerous']
    dysfunction_count = mechanical_conn[mechanical_conn['mechanism_function'].isin(dysfunctional)].shape[0]
    normal_count = mechanical_conn[mechanical_conn['mechanism_function'].isin(['working_normal', 'working_slow', 'working_fast'])].shape[0]
    
    if dysfunction_count + normal_count > 0:
        dysfunction_rate = dysfunction_count / (dysfunction_count + normal_count) * 100
        print(f"\n  ✓ DYSFUNCTION RATE: {dysfunction_rate:.1f}%")
        print(f"    ({dysfunction_count} dysfunctional vs {normal_count} functional)")
        
        # One-sample binomial test against 50% (random chance) - using binomtest (new API)
        from scipy.stats import binomtest
        total = dysfunction_count + normal_count
        result_binom = binomtest(dysfunction_count, total, 0.5, alternative='greater')
        print(f"\n  Binomial test (H0: dysfunction rate = 50%):")
        print(f"    p = {result_binom.pvalue:.4f}")
        if result_binom.pvalue < 0.05:
            print("    ⚠️ SIGNIFICANT: Elevators malfunction MORE than chance")
        else:
            print("    ✓ Not significant: dysfunction rate NOT above 50%")
else:
    print("  ⚠️ Insufficient mechanical connection data")


1.2 MECHANICAL CONNECTION DYSFUNCTION
--------------------------------------------------
  Total mechanical connections (elevators/escalators): 289

  Mechanism function distribution:
    working_normal           : 128 (44.3%)
    not_mentioned            : 66 (22.8%)
    unpredictable            : 51 (17.6%)
    dangerous                : 18 (6.2%)
    working_fast             : 11 (3.8%)
    broken                   : 4 (1.4%)
    malfunctioning           : 4 (1.4%)
    not_applicable           : 4 (1.4%)
    working_slow             : 3 (1.0%)

  ✓ DYSFUNCTION RATE: 35.2%
    (77 dysfunctional vs 142 functional)

  Binomial test (H0: dysfunction rate = 50%):
    p = 1.0000
    ✓ Not significant: dysfunction rate NOT above 50%


---

# HYPOTHESIS 2: Somatic Sphere Conflict

**Swedenborgian Grounding:** When a spirit approaches a society with a contrary sphere, it causes physical distress - "anxieties... caused by reciprocal aversions" manifesting as breathing difficulty, swooning, or twisting sensations.

**Predictions:**
1. NAUSEA, DIZZINESS, PARALYSIS cluster in locations with COLD_WHITE light or OPPRESSIVE atmosphere
2. COMFORT correlates with WARM_GOLDEN light
3. GLITCHING represents cognitive dissonance of the "external mind" rejecting "internal reality"

In [13]:
print("="*70)
print("HYPOTHESIS 2: SOMATIC SPHERE CONFLICT")
print("="*70)

# Test 2.1: Somatic Response vs Light Temperature
print("\n2.1 SOMATIC RESPONSE × LIGHT TEMPERATURE")
print("-" * 50)

# Filter to locations with both fields populated
somatic_light = df_locations[
    (df_locations['somatic_response'] != 'none') &
    (df_locations['light_temperature'] != 'not_mentioned')
].copy()

print(f"  Locations with somatic response AND light temperature: {len(somatic_light):,}")

if len(somatic_light) >= 20:
    contingency = pd.crosstab(somatic_light['somatic_response'], somatic_light['light_temperature'])
    print(f"\n  Contingency table:")
    print(contingency)
    
    if contingency.shape[0] > 1 and contingency.shape[1] > 1:
        result = chi_square_test(contingency, "Somatic Response vs Light Temperature")
        report_chi_square(result)
else:
    print("  ⚠️ Insufficient co-occurrence data")
    print("\n  Examining each variable separately:")
    
    # Somatic response distribution
    somatic_dist = df_locations[df_locations['somatic_response'] != 'none']['somatic_response'].value_counts()
    print(f"\n  Somatic responses (n={somatic_dist.sum()}):")
    for resp, count in somatic_dist.items():
        print(f"    {resp:20s}: {count:,}")
    
    # Light temperature distribution
    light_temp_dist = df_locations[df_locations['light_temperature'] != 'not_mentioned']['light_temperature'].value_counts()
    print(f"\n  Light temperatures (n={light_temp_dist.sum()}):")
    for temp, count in light_temp_dist.items():
        print(f"    {temp:20s}: {count:,}")

HYPOTHESIS 2: SOMATIC SPHERE CONFLICT

2.1 SOMATIC RESPONSE × LIGHT TEMPERATURE
--------------------------------------------------
  Locations with somatic response AND light temperature: 43

  Contingency table:
light_temperature  cold_white  neutral  warm_golden
somatic_response                                   
comfort                     2        0           11
dizziness                   2        0            4
ejection                    3        0            1
glitching                   2        1            0
heaviness                   3        0            3
nausea                      1        0            0
paralysis                   4        0            1
sleepiness                  2        0            3

Somatic Response vs Light Temperature
--------------------------------------------------
  n = 43
  χ² = 25.76, df = 14, p = 2.7795e-02
  Cramér's V = 0.547 (large effect)
  ⚠️ WEAKLY SIGNIFICANT (p < 0.05)


In [14]:
# Test 2.2: Somatic Response vs Atmosphere
print("\n2.2 SOMATIC RESPONSE × ATMOSPHERE")
print("-" * 50)

somatic_atmos = df_locations[
    (df_locations['somatic_response'] != 'none') &
    (df_locations['atmosphere'] != 'not_mentioned')
].copy()

print(f"  Locations with somatic response AND atmosphere: {len(somatic_atmos):,}")

if len(somatic_atmos) >= 20:
    contingency = pd.crosstab(somatic_atmos['somatic_response'], somatic_atmos['atmosphere'])
    print(f"\n  Contingency table:")
    print(contingency)
    
    if contingency.shape[0] > 1 and contingency.shape[1] > 1:
        result = chi_square_test(contingency, "Somatic Response vs Atmosphere")
        report_chi_square(result)
        
        # Specific check: distress symptoms in OPPRESSIVE atmospheres
        distress_symptoms = ['nausea', 'dizziness', 'paralysis', 'heaviness', 'glitching']
        if 'oppressive' in contingency.columns:
            oppressive_distress = contingency.loc[contingency.index.isin(distress_symptoms), 'oppressive'].sum()
            total_oppressive = contingency['oppressive'].sum()
            if total_oppressive > 0:
                print(f"\n  Distress symptoms in OPPRESSIVE locations: {oppressive_distress}/{total_oppressive} ({oppressive_distress/total_oppressive*100:.1f}%)")
else:
    print("  ⚠️ Insufficient co-occurrence data")


2.2 SOMATIC RESPONSE × ATMOSPHERE
--------------------------------------------------
  Locations with somatic response AND atmosphere: 203

  Contingency table:
atmosphere        chaotic  eerie  neutral  nostalgic  oppressive  peaceful  \
somatic_response                                                             
comfort                 0      0        1          1           0         9   
dizziness               0      2        1          0           2         0   
ejection                4     10        8          2           7         0   
glitching               2      8        0          0           6         0   
headache                0      0        0          0           1         0   
heaviness               1      3        1          0           7         1   
nausea                  2      0        1          0           2         0   
paralysis               0      1        2          0           5         0   
sleepiness              1      1        6          0      

In [15]:
# Test 2.3: Overall somatic response patterns
print("\n2.3 SOMATIC RESPONSE PATTERNS (Overall)")
print("-" * 50)

# Full distribution
somatic_all = df_locations['somatic_response'].value_counts()
print(f"  All somatic responses (n={len(df_locations)}):")
for resp, count in somatic_all.items():
    pct = count / len(df_locations) * 100
    print(f"    {resp:20s}: {count:,} ({pct:.1f}%)")

# Non-'none' responses
has_somatic = (df_locations['somatic_response'] != 'none').sum()
print(f"\n  Locations with ANY somatic response: {has_somatic:,} ({has_somatic/len(df_locations)*100:.1f}%)")

# Distress vs comfort
distress_types = ['nausea', 'dizziness', 'paralysis', 'heaviness', 'headache', 'sleepiness', 'glitching', 'ejection']
comfort_types = ['comfort']

distress_count = df_locations[df_locations['somatic_response'].isin(distress_types)].shape[0]
comfort_count = df_locations[df_locations['somatic_response'].isin(comfort_types)].shape[0]

print(f"\n  DISTRESS responses: {distress_count:,}")
print(f"  COMFORT responses: {comfort_count:,}")
if distress_count + comfort_count > 0:
    print(f"  Distress:Comfort ratio: {distress_count}:{comfort_count}")


2.3 SOMATIC RESPONSE PATTERNS (Overall)
--------------------------------------------------
  All somatic responses (n=8707):
    none                : 8,443 (97.0%)
    ejection            : 90 (1.0%)
    glitching           : 33 (0.4%)
    comfort             : 30 (0.3%)
    paralysis           : 29 (0.3%)
    heaviness           : 29 (0.3%)
    sleepiness          : 28 (0.3%)
    dizziness           : 14 (0.2%)
    nausea              : 8 (0.1%)
    headache            : 3 (0.0%)

  Locations with ANY somatic response: 264 (3.0%)

  DISTRESS responses: 234
  COMFORT responses: 30
  Distress:Comfort ratio: 234:30


---

# HYPOTHESIS 3: The Digestive Sorting Mechanism

**Swedenborgian Grounding:** The World of Spirits corresponds to the digestive system of the Grand Man. Its function is to "break down" the spirit, separating "nutritious" parts (truths/goods) from "excrementitious" parts (evils/falsities).

**Predictions:**
1. High prevalence of PROCESSING locations: SCHOOL, HOSPITAL, AIRPORT
2. In schools: TESTING or ARCHIVAL focus (not PRACTICAL learning)
3. Authority figures behave as OBSERVING or GUIDING (sorting)

In [16]:
print("="*70)
print("HYPOTHESIS 3: DIGESTIVE SORTING MECHANISM")
print("="*70)

# Test 3.1: Location type distribution
print("\n3.1 LOCATION TYPE DISTRIBUTION")
print("-" * 50)

# Define processing/sorting categories
PROCESSING_LOCATIONS = {
    'school', 'school_classroom', 'school_gym', 'school_cafeteria', 'school_library',
    'hospital', 'hospital_room', 'hospital_waiting',
    'airport', 'airport_terminal', 'airport_gate', 'airport_security',
    'train_station', 'bus_station', 'subway'
}

WASTE_LOCATIONS = {
    'bathroom', 'bathroom_public', 'bathroom_locker_room',
    'basement', 'underground'
}

COMMERCIAL_LOCATIONS = {
    'mall', 'mall_food_court', 'mall_store', 'mall_arcade', 'mall_theater',
    'casino', 'restaurant', 'restaurant_fast_food', 'restaurant_buffet'
}

# Categorize all locations
def categorize_location(loc_type):
    if loc_type in PROCESSING_LOCATIONS:
        return 'processing'
    elif loc_type in WASTE_LOCATIONS:
        return 'waste'
    elif loc_type in COMMERCIAL_LOCATIONS:
        return 'commercial'
    else:
        return 'other'

df_locations['location_category'] = df_locations['location_type'].apply(categorize_location)

# Distribution
category_dist = df_locations['location_category'].value_counts()
print(f"  Location category distribution (n={len(df_locations)}):")
for cat, count in category_dist.items():
    print(f"    {cat:15s}: {count:,} ({count/len(df_locations)*100:.1f}%)")

# Detailed breakdown of processing locations
processing_locs = df_locations[df_locations['location_category'] == 'processing']
print(f"\n  PROCESSING location breakdown (n={len(processing_locs)}):")
processing_types = processing_locs['location_type'].value_counts()
for loc_type, count in processing_types.items():
    print(f"    {loc_type:25s}: {count:,}")

HYPOTHESIS 3: DIGESTIVE SORTING MECHANISM

3.1 LOCATION TYPE DISTRIBUTION
--------------------------------------------------
  Location category distribution (n=8707):
    other          : 5,557 (63.8%)
    commercial     : 2,067 (23.7%)
    processing     : 749 (8.6%)
    waste          : 334 (3.8%)

  PROCESSING location breakdown (n=749):
    school                   : 264
    airport                  : 116
    train_station            : 66
    hospital                 : 60
    school_classroom         : 59
    subway                   : 41
    school_cafeteria         : 30
    school_gym               : 28
    airport_terminal         : 25
    hospital_waiting         : 16
    school_library           : 15
    bus_station              : 10
    hospital_room            : 9
    airport_gate             : 7
    airport_security         : 3


In [17]:
# Test 3.2: Intellectual Focus in Schools
print("\n3.2 INTELLECTUAL FOCUS IN SCHOOLS")
print("-" * 50)

# Filter to school locations
school_types = ['school', 'school_classroom', 'school_gym', 'school_cafeteria', 'school_library']
school_locs = df_locations[df_locations['location_type'].isin(school_types)].copy()

print(f"  Total school locations: {len(school_locs):,}")

if len(school_locs) > 0:
    focus_dist = school_locs['intellectual_focus'].value_counts()
    print(f"\n  Intellectual focus in schools:")
    for focus, count in focus_dist.items():
        print(f"    {focus:20s}: {count:,} ({count/len(school_locs)*100:.1f}%)")
    
    # Prediction check: TESTING or ARCHIVAL should be more common than PRACTICAL
    testing_archival = school_locs[school_locs['intellectual_focus'].isin(['testing', 'archival'])].shape[0]
    practical = school_locs[school_locs['intellectual_focus'] == 'practical'].shape[0]
    
    if testing_archival + practical > 0:
        print(f"\n  Prediction check:")
        print(f"    TESTING/ARCHIVAL: {testing_archival}")
        print(f"    PRACTICAL: {practical}")
        if testing_archival > practical:
            print("    ✓ SUPPORTS hypothesis: Testing/Archival > Practical")
        elif testing_archival < practical:
            print("    ✗ CONTRADICTS hypothesis: Practical > Testing/Archival")
        else:
            print("    = Equal distribution")


3.2 INTELLECTUAL FOCUS IN SCHOOLS
--------------------------------------------------
  Total school locations: 396

  Intellectual focus in schools:
    not_mentioned       : 286 (72.2%)
    practical           : 52 (13.1%)
    testing             : 26 (6.6%)
    obscured            : 16 (4.0%)
    archival            : 15 (3.8%)
    argumentative       : 1 (0.3%)

  Prediction check:
    TESTING/ARCHIVAL: 41
    PRACTICAL: 52
    ✗ CONTRADICTS hypothesis: Practical > Testing/Archival


In [18]:
# Test 3.3: Authority Figure Nature
print("\n3.3 AUTHORITY FIGURE BEHAVIOR")
print("-" * 50)

if len(df_entities) > 0:
    # Filter to authority figures
    authority_roles = ['security', 'teacher', 'staff', 'authority']
    authority_entities = df_entities[
        (df_entities['role'].isin(authority_roles)) | 
        (df_entities['entity_type'] == 'authority')
    ].copy()
    
    print(f"  Total authority figure entities: {len(authority_entities):,}")
    
    if len(authority_entities) > 0:
        auth_nature_dist = authority_entities['authority_nature'].value_counts()
        print(f"\n  Authority nature distribution:")
        for nature, count in auth_nature_dist.items():
            print(f"    {nature:20s}: {count:,} ({count/len(authority_entities)*100:.1f}%)")
        
        # Prediction: OBSERVING and GUIDING should be common
        sorting_behaviors = ['observing', 'guiding']
        sorting_count = authority_entities[authority_entities['authority_nature'].isin(sorting_behaviors)].shape[0]
        punitive_count = authority_entities[authority_entities['authority_nature'] == 'punitive'].shape[0]
        
        print(f"\n  Prediction check:")
        print(f"    OBSERVING/GUIDING (sorting): {sorting_count}")
        print(f"    PUNITIVE: {punitive_count}")
else:
    print("  ⚠️ No entity data available")


3.3 AUTHORITY FIGURE BEHAVIOR
--------------------------------------------------
  Total authority figure entities: 661

  Authority nature distribution:
    not_mentioned       : 182 (27.5%)
    blocking            : 108 (16.3%)
    observing           : 106 (16.0%)
    punitive            : 101 (15.3%)
    guiding             : 101 (15.3%)
    pursuing            : 63 (9.5%)

  Prediction check:
    OBSERVING/GUIDING (sorting): 207
    PUNITIVE: 101


---

# HYPOTHESIS 4: The Exposure of the Proprium (Bathroom Archetype)

**Swedenborgian Grounding:** "Excrement" corresponds to the "Love of Self" (the Proprium). When external bonds of social politeness are stripped away, the "filthy" nature of these loves is revealed. Hellish caverns are described as "privies."

**Predictions:**
1. BATHROOM locations correlate with EXPOSED or CROWDED_EXPOSURE privacy status
2. BATHROOM cleanliness is typically FILTHY or DISGUSTING
3. Clean + Private bathroom = "Closed Internal" (mask intact)
4. Filthy + Exposed = witnessing actual state of hellish Proprium

In [20]:
print("="*70)
print("HYPOTHESIS 4: EXPOSURE OF THE PROPRIUM")
print("="*70)

# Test 4.1: Bathroom Privacy Status
print("\n4.1 BATHROOM PRIVACY STATUS")
print("-" * 50)

bathroom_types = ['bathroom', 'bathroom_public', 'bathroom_locker_room']
bathroom_locs = df_locations[df_locations['location_type'].isin(bathroom_types)].copy()

print(f"  Total bathroom locations: {len(bathroom_locs):,}")

if len(bathroom_locs) > 10:
    privacy_dist = bathroom_locs['privacy_status'].value_counts()
    print(f"\n  Privacy status in bathrooms:")
    for status, count in privacy_dist.items():
        print(f"    {status:25s}: {count:,} ({count/len(bathroom_locs)*100:.1f}%)")
    
    # Prediction: EXPOSED categories should dominate
    exposed_categories = ['exposed', 'compromised', 'crowded_exposure']
    private_categories = ['private', 'natural_seclusion']
    
    exposed_count = bathroom_locs[bathroom_locs['privacy_status'].isin(exposed_categories)].shape[0]
    private_count = bathroom_locs[bathroom_locs['privacy_status'].isin(private_categories)].shape[0]
    not_mentioned = bathroom_locs[bathroom_locs['privacy_status'] == 'not_mentioned'].shape[0]
    
    print(f"\n  Prediction check:")
    print(f"    EXPOSED categories: {exposed_count}")
    print(f"    PRIVATE categories: {private_count}")
    print(f"    Not mentioned: {not_mentioned}")
    
    # Statistical test (excluding not_mentioned)
    if exposed_count + private_count > 10:
        # Binomial test: is exposed rate > 50%?
        total = exposed_count + private_count
        exposed_rate = exposed_count / total
        result_binom = stats.binomtest(exposed_count, total, 0.5, alternative='greater')
        print(f"\n  Exposed rate: {exposed_rate*100:.1f}%")
        print(f"  Binomial test (H0: exposed rate = 50%): p = {result_binom.pvalue:.4f}")
        if result_binom.pvalue < 0.05:
            print("  ⚠️ SIGNIFICANT: Bathrooms show EXPOSURE more than chance")
        else:
            print("  ✓ Not significant")
else:
    print("  ⚠️ Insufficient bathroom data")

HYPOTHESIS 4: EXPOSURE OF THE PROPRIUM

4.1 BATHROOM PRIVACY STATUS
--------------------------------------------------
  Total bathroom locations: 207

  Privacy status in bathrooms:
    not_mentioned            : 126 (60.9%)
    compromised              : 46 (22.2%)
    exposed                  : 17 (8.2%)
    private                  : 13 (6.3%)
    crowded_exposure         : 5 (2.4%)

  Prediction check:
    EXPOSED categories: 68
    PRIVATE categories: 13
    Not mentioned: 126

  Exposed rate: 84.0%
  Binomial test (H0: exposed rate = 50%): p = 0.0000
  ⚠️ SIGNIFICANT: Bathrooms show EXPOSURE more than chance


In [21]:
# Test 4.2: Bathroom Cleanliness
print("\n4.2 BATHROOM CLEANLINESS")
print("-" * 50)

if len(bathroom_locs) > 10:
    cleanliness_dist = bathroom_locs['cleanliness'].value_counts()
    print(f"  Cleanliness in bathrooms:")
    for status, count in cleanliness_dist.items():
        print(f"    {status:20s}: {count:,} ({count/len(bathroom_locs)*100:.1f}%)")
    
    # Prediction: FILTHY/DISGUSTING should be common
    filthy_categories = ['filthy', 'disgusting', 'dirty']
    clean_categories = ['sterile', 'clean', 'normal']
    
    filthy_count = bathroom_locs[bathroom_locs['cleanliness'].isin(filthy_categories)].shape[0]
    clean_count = bathroom_locs[bathroom_locs['cleanliness'].isin(clean_categories)].shape[0]
    
    print(f"\n  Prediction check:")
    print(f"    FILTHY/DISGUSTING/DIRTY: {filthy_count}")
    print(f"    CLEAN/STERILE/NORMAL: {clean_count}")
    
    if filthy_count + clean_count > 10:
        filthy_rate = filthy_count / (filthy_count + clean_count)
        result_binom = stats.binomtest(filthy_count, filthy_count + clean_count, 0.5, alternative='greater')
        print(f"\n  Filthy rate: {filthy_rate*100:.1f}%")
        print(f"  Binomial test: p = {result_binom.pvalue:.4f}")
else:
    print("  ⚠️ Insufficient bathroom data")


4.2 BATHROOM CLEANLINESS
--------------------------------------------------
  Cleanliness in bathrooms:
    not_mentioned       : 127 (61.4%)
    dirty               : 29 (14.0%)
    disgusting          : 26 (12.6%)
    clean               : 16 (7.7%)
    filthy              : 7 (3.4%)
    sterile             : 2 (1.0%)

  Prediction check:
    FILTHY/DISGUSTING/DIRTY: 62
    CLEAN/STERILE/NORMAL: 18

  Filthy rate: 77.5%
  Binomial test: p = 0.0000


In [22]:
# Test 4.3: Privacy × Cleanliness Cross-tabulation
print("\n4.3 PRIVACY × CLEANLINESS PATTERN")
print("-" * 50)

if len(bathroom_locs) > 10:
    # Create binary categories for cleaner analysis
    bathroom_analysis = bathroom_locs[
        (bathroom_locs['privacy_status'] != 'not_mentioned') &
        (bathroom_locs['cleanliness'] != 'not_mentioned')
    ].copy()
    
    print(f"  Bathrooms with both privacy AND cleanliness data: {len(bathroom_analysis)}")
    
    if len(bathroom_analysis) >= 10:
        # Create binary variables
        bathroom_analysis['is_exposed'] = bathroom_analysis['privacy_status'].isin(['exposed', 'compromised', 'crowded_exposure'])
        bathroom_analysis['is_filthy'] = bathroom_analysis['cleanliness'].isin(['filthy', 'disgusting', 'dirty'])
        
        # Cross-tabulation
        contingency = pd.crosstab(bathroom_analysis['is_exposed'], bathroom_analysis['is_filthy'])
        contingency.index = ['Private', 'Exposed']
        contingency.columns = ['Clean', 'Filthy']
        print(f"\n  Cross-tabulation:")
        print(contingency)
        
        # Chi-square or Fisher's exact
        if contingency.shape == (2, 2):
            if contingency.min().min() < 5:
                # Fisher's exact for small samples
                odds, p = fisher_exact(contingency)
                print(f"\n  Fisher's exact test:")
                print(f"    Odds ratio = {odds:.2f}")
                print(f"    p = {p:.4f}")
            else:
                result = chi_square_test(contingency, "Privacy × Cleanliness")
                report_chi_square(result)
            
            # Calculate pattern percentages
            total = contingency.sum().sum()
            exposed_filthy = contingency.loc['Exposed', 'Filthy'] if 'Exposed' in contingency.index and 'Filthy' in contingency.columns else 0
            private_clean = contingency.loc['Private', 'Clean'] if 'Private' in contingency.index and 'Clean' in contingency.columns else 0
            
            print(f"\n  Pattern percentages:")
            print(f"    Exposed + Filthy (Proprium revealed): {exposed_filthy} ({exposed_filthy/total*100:.1f}%)")
            print(f"    Private + Clean (Mask intact): {private_clean} ({private_clean/total*100:.1f}%)")
else:
    print("  ⚠️ Insufficient bathroom data")


4.3 PRIVACY × CLEANLINESS PATTERN
--------------------------------------------------
  Bathrooms with both privacy AND cleanliness data: 45

  Cross-tabulation:
         Clean  Filthy
Private      4       2
Exposed      6      33

  Fisher's exact test:
    Odds ratio = 11.00
    p = 0.0164

  Pattern percentages:
    Exposed + Filthy (Proprium revealed): 33 (73.3%)
    Private + Clean (Mask intact): 4 (8.9%)


---

# HYPOTHESIS 5: The "Plastic" Heavens

**Swedenborgian Grounding:** Swedenborg describes "Phantasies" where spirits create magnificent palaces that are merely projections of vanity. These structures are unstable; when the "light of truth" flows in, they dissolve into "decaying" ruins.

**Predictions:**
1. CASINO, MALL_STORE, HOTEL_BALLROOM locations show PLASTIC or SHIFTING reality stability
2. Luxury locations have DECAYING state beneath the surface
3. SOLID reality found mainly in FOREST/NATURE or simple/humble structures

In [23]:
print("="*70)
print("HYPOTHESIS 5: PLASTIC HEAVENS")
print("="*70)

# Test 5.1: Reality Stability by Location Category
print("\n5.1 REALITY STABILITY × LOCATION TYPE")
print("-" * 50)

# Define luxury/vanity locations
LUXURY_LOCATIONS = {
    'casino', 'mall_store', 'hotel_ballroom', 'hotel_lobby', 'mansion',
    'mall', 'restaurant_buffet', 'amusement_park'
}

NATURE_LOCATIONS = {
    'forest', 'mountain', 'beach', 'island', 'cave'
}

HUMBLE_LOCATIONS = {
    'house', 'apartment', 'dorm', 'neighborhood'
}

def categorize_luxury(loc_type):
    if loc_type in LUXURY_LOCATIONS:
        return 'luxury'
    elif loc_type in NATURE_LOCATIONS:
        return 'nature'
    elif loc_type in HUMBLE_LOCATIONS:
        return 'humble'
    else:
        return 'other'

df_locations['luxury_category'] = df_locations['location_type'].apply(categorize_luxury)

# Reality stability distribution by category
for category in ['luxury', 'nature', 'humble']:
    cat_locs = df_locations[df_locations['luxury_category'] == category]
    print(f"\n  {category.upper()} locations (n={len(cat_locs)}):")
    if len(cat_locs) > 0:
        reality_dist = cat_locs['reality_stability'].value_counts()
        for stability, count in reality_dist.items():
            print(f"    {stability:20s}: {count:,} ({count/len(cat_locs)*100:.1f}%)")

HYPOTHESIS 5: PLASTIC HEAVENS

5.1 REALITY STABILITY × LOCATION TYPE
--------------------------------------------------

  LUXURY locations (n=1805):
    not_mentioned       : 1,017 (56.3%)
    solid               : 628 (34.8%)
    shifting            : 107 (5.9%)
    hyper_real          : 43 (2.4%)
    plastic             : 8 (0.4%)
    decaying            : 2 (0.1%)

  NATURE locations (n=397):
    not_mentioned       : 202 (50.9%)
    solid               : 166 (41.8%)
    hyper_real          : 14 (3.5%)
    shifting            : 13 (3.3%)
    plastic             : 2 (0.5%)

  HUMBLE locations (n=646):
    not_mentioned       : 332 (51.4%)
    solid               : 212 (32.8%)
    shifting            : 77 (11.9%)
    hyper_real          : 18 (2.8%)
    plastic             : 7 (1.1%)


In [24]:
# Test 5.2: Statistical comparison of reality stability
print("\n5.2 LUXURY vs OTHER: REALITY STABILITY")
print("-" * 50)

# Filter to locations with reality stability data
stability_data = df_locations[
    (df_locations['reality_stability'] != 'not_mentioned') &
    (df_locations['luxury_category'].isin(['luxury', 'nature', 'humble', 'other']))
].copy()

print(f"  Locations with reality stability data: {len(stability_data)}")

if len(stability_data) >= 30:
    # Create binary: luxury vs non-luxury
    stability_data['is_luxury'] = stability_data['luxury_category'] == 'luxury'
    
    # Create binary: unstable vs stable
    unstable_types = ['plastic', 'shifting', 'decaying']
    stable_types = ['solid', 'hyper_real']
    stability_data['is_unstable'] = stability_data['reality_stability'].isin(unstable_types)
    
    # Contingency table
    contingency = pd.crosstab(stability_data['is_luxury'], stability_data['is_unstable'])
    contingency.index = ['Non-Luxury', 'Luxury']
    contingency.columns = ['Stable', 'Unstable']
    print(f"\n  Contingency table:")
    print(contingency)
    
    if contingency.shape == (2, 2) and contingency.sum().sum() > 10:
        if contingency.min().min() < 5:
            odds, p = fisher_exact(contingency)
            print(f"\n  Fisher's exact test:")
            print(f"    Odds ratio = {odds:.2f}")
            print(f"    p = {p:.4f}")
            if p < 0.05 and odds > 1:
                print("    ⚠️ SIGNIFICANT: Luxury locations show MORE instability")
            elif p < 0.05 and odds < 1:
                print("    ✗ CONTRADICTS: Luxury locations show LESS instability")
            else:
                print("    ✓ Not significant")
        else:
            result = chi_square_test(contingency, "Luxury × Reality Stability")
            report_chi_square(result)
else:
    print("  ⚠️ Insufficient reality stability data")


5.2 LUXURY vs OTHER: REALITY STABILITY
--------------------------------------------------
  Locations with reality stability data: 3804

  Contingency table:
            Stable  Unstable
Non-Luxury    2457       559
Luxury         671       117

Luxury × Reality Stability
--------------------------------------------------
  n = 3,804
  χ² = 5.56, df = 1, p = 1.8357e-02
  Cramér's V = 0.038 (negligible effect)
  ⚠️ WEAKLY SIGNIFICANT (p < 0.05)


In [25]:
# Test 5.3: State of Place (Decaying) by Location Category
print("\n5.3 STATE OF PLACE BY CATEGORY")
print("-" * 50)

state_data = df_locations[
    (df_locations['state'] != 'not_mentioned') &
    (df_locations['luxury_category'].isin(['luxury', 'nature', 'humble']))
].copy()

print(f"  Locations with state data: {len(state_data)}")

if len(state_data) >= 20:
    # Distribution by category
    for category in ['luxury', 'nature', 'humble']:
        cat_data = state_data[state_data['luxury_category'] == category]
        if len(cat_data) > 0:
            print(f"\n  {category.upper()} (n={len(cat_data)}):")
            state_dist = cat_data['state'].value_counts()
            for state, count in state_dist.items():
                print(f"    {state:20s}: {count:,}")
    
    # Prediction: Luxury should have higher DECAYING rate
    decay_states = ['decaying', 'abandoned', 'destroyed']
    
    luxury_decay = state_data[(state_data['luxury_category'] == 'luxury') & 
                              (state_data['state'].isin(decay_states))].shape[0]
    luxury_total = state_data[state_data['luxury_category'] == 'luxury'].shape[0]
    
    other_decay = state_data[(state_data['luxury_category'] != 'luxury') & 
                             (state_data['state'].isin(decay_states))].shape[0]
    other_total = state_data[state_data['luxury_category'] != 'luxury'].shape[0]
    
    if luxury_total > 0 and other_total > 0:
        print(f"\n  Decay rate comparison:")
        print(f"    LUXURY decay rate: {luxury_decay}/{luxury_total} ({luxury_decay/luxury_total*100:.1f}%)")
        print(f"    OTHER decay rate: {other_decay}/{other_total} ({other_decay/other_total*100:.1f}%)")


5.3 STATE OF PLACE BY CATEGORY
--------------------------------------------------
  Locations with state data: 960

  LUXURY (n=608):
    maintained          : 398
    mixed               : 67
    worn                : 55
    abandoned           : 41
    new                 : 17
    decaying            : 15
    under_construction  : 7
    destroyed           : 6
    renovating          : 2

  NATURE (n=101):
    maintained          : 81
    mixed               : 7
    decaying            : 5
    destroyed           : 4
    abandoned           : 2
    worn                : 2

  HUMBLE (n=251):
    maintained          : 121
    worn                : 46
    mixed               : 28
    decaying            : 20
    abandoned           : 17
    new                 : 6
    destroyed           : 5
    under_construction  : 4
    renovating          : 4

  Decay rate comparison:
    LUXURY decay rate: 62/608 (10.2%)
    OTHER decay rate: 53/352 (15.1%)


---

# SUMMARY: Results Overview

In [26]:
print("="*70)
print("HYPOTHESIS TESTING SUMMARY")
print("="*70)

print("""
HYPOTHESIS 1: Affectional Proximity (Non-Locality)
─────────────────────────────────────────────────
Prediction: Travel depends on emotional state, not physical locomotion
Key markers: TransitMode × TraversalOutcome, MechanismFunction
Status: [See detailed results above]

HYPOTHESIS 2: Somatic Sphere Conflict
─────────────────────────────────────────────────
Prediction: Physical distress correlates with "incompatible" environments
Key markers: SomaticResponse × LightTemperature × Atmosphere
Status: [See detailed results above]

HYPOTHESIS 3: Digestive Sorting Mechanism
─────────────────────────────────────────────────
Prediction: High prevalence of processing/sorting locations
Key markers: LocationType distribution, IntellectualFocus in schools
Status: [See detailed results above]

HYPOTHESIS 4: Exposure of the Proprium
─────────────────────────────────────────────────
Prediction: Bathrooms show exposure and filth patterns
Key markers: PrivacyStatus × Cleanliness in bathrooms
Status: [See detailed results above]

HYPOTHESIS 5: Plastic Heavens
─────────────────────────────────────────────────
Prediction: Luxury locations exhibit reality instability
Key markers: RealityStability × LocationType, StateOfPlace
Status: [See detailed results above]
""")

print("\n" + "="*70)
print("DATA LIMITATIONS")
print("="*70)
print(f"""
1. Sample size: {len(df_dreams):,} valid dream reports
2. Many fields have high 'not_mentioned' rates (see quality assessment)
3. Connection data may be sparse relative to location data
4. Entity data depends on interactions being recorded
5. Reality stability and light temperature require explicit mention
""")

HYPOTHESIS TESTING SUMMARY

HYPOTHESIS 1: Affectional Proximity (Non-Locality)
─────────────────────────────────────────────────
Prediction: Travel depends on emotional state, not physical locomotion
Key markers: TransitMode × TraversalOutcome, MechanismFunction
Status: [See detailed results above]

HYPOTHESIS 2: Somatic Sphere Conflict
─────────────────────────────────────────────────
Prediction: Physical distress correlates with "incompatible" environments
Key markers: SomaticResponse × LightTemperature × Atmosphere
Status: [See detailed results above]

HYPOTHESIS 3: Digestive Sorting Mechanism
─────────────────────────────────────────────────
Prediction: High prevalence of processing/sorting locations
Key markers: LocationType distribution, IntellectualFocus in schools
Status: [See detailed results above]

HYPOTHESIS 4: Exposure of the Proprium
─────────────────────────────────────────────────
Prediction: Bathrooms show exposure and filth patterns
Key markers: PrivacyStatus × Cleanl

---

# APPENDIX: Additional Exploratory Analysis

In [ ]:
# Visualization: Location Type Distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Top 15 location types
ax1 = axes[0]
loc_type_top = df_locations['location_type'].value_counts().head(15)
colors = ['#FF6B6B' if lt in PROCESSING_LOCATIONS else 
          '#4ECDC4' if lt in WASTE_LOCATIONS else
          '#95E1D3' if lt in COMMERCIAL_LOCATIONS else
          '#B5B5B5' for lt in loc_type_top.index]
bars = ax1.barh(loc_type_top.index[::-1], loc_type_top.values[::-1], color=colors[::-1])
ax1.set_xlabel('Count')
ax1.set_title('Top 15 Location Types\n(Red=Processing, Teal=Waste, Green=Commercial)', fontsize=12)

# Panel 2: Category distribution
ax2 = axes[1]
cat_dist = df_locations['location_category'].value_counts()
cat_colors = {'processing': '#FF6B6B', 'waste': '#4ECDC4', 'commercial': '#95E1D3', 'other': '#B5B5B5'}
colors = [cat_colors.get(cat, '#B5B5B5') for cat in cat_dist.index]
ax2.pie(cat_dist.values, labels=cat_dist.index, colors=colors, autopct='%1.1f%%', startangle=90)
ax2.set_title('Location Category Distribution', fontsize=12)

plt.tight_layout()
plt.savefig('../reports/figures/location_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved to ../reports/figures/location_distribution.png")

In [27]:
# Final data export for further analysis
print("\nExporting analysis datasets...")

# Create reports/data directory if needed
output_dir = Path('../reports/data')
output_dir.mkdir(parents=True, exist_ok=True)

# Export key dataframes
df_dreams.to_csv(output_dir / 'dreams_meta.csv', index=False)
df_locations.to_csv(output_dir / 'locations.csv', index=False)
df_connections.to_csv(output_dir / 'connections.csv', index=False)
df_interactions.to_csv(output_dir / 'interactions.csv', index=False)
df_entities.to_csv(output_dir / 'entities.csv', index=False)

print(f"✓ Exported to {output_dir}:")
print(f"  - dreams_meta.csv ({len(df_dreams):,} rows)")
print(f"  - locations.csv ({len(df_locations):,} rows)")
print(f"  - connections.csv ({len(df_connections):,} rows)")
print(f"  - interactions.csv ({len(df_interactions):,} rows)")
print(f"  - entities.csv ({len(df_entities):,} rows)")


Exporting analysis datasets...
✓ Exported to ..\reports\data:
  - dreams_meta.csv (1,926 rows)
  - locations.csv (8,707 rows)
  - connections.csv (4,600 rows)
  - interactions.csv (6,075 rows)
  - entities.csv (3,704 rows)
